# Self-Play Risiko v8

Training generazionale: il bot impara giocando contro versioni precedenti di sé stesso.

## Storia
- Baseline 29% (vs random) raggiunto.
- Stage A v1: 19% (peggio della baseline, fallito)
- Stage A2: feature riprogettate, non testato in self-play
- **v8**: cambiamo paradigma. Niente più training vs random — self-play.

## Cella 1 — Setup

In [ ]:
# Clone repo se non esiste
import os
if not os.path.exists('/content/risiko-rl'):
    !git clone https://github.com/EdoMusu1991/risiko-rl.git /content/risiko-rl
%cd /content/risiko-rl
!git pull

import sys
sys.path.insert(0, '/content/risiko-rl')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

CHECKPOINT_DIR = '/content/drive/MyDrive/risiko-rl-checkpoints'
POPULATION_DIR = f'{CHECKPOINT_DIR}/population_v8'
BASELINE_PATH = f'{CHECKPOINT_DIR}/baseline_models/v4test_500k_baseline_29wr.zip'

import os
os.makedirs(POPULATION_DIR, exist_ok=True)

# Verifica baseline esistente
assert os.path.exists(BASELINE_PATH), f'Baseline non trovata in {BASELINE_PATH}. Caricala prima.'
print(f'Baseline 29% trovata: {BASELINE_PATH}')
print(f'Population dir: {POPULATION_DIR}')

In [ ]:
!pip install -q stable-baselines3 sb3-contrib gymnasium

## Cella 2 — Configurazione self-play

In [ ]:
# Configurazione
N_GENERAZIONI = 4         # quante generazioni di self-play
STEPS_PER_GENERAZIONE = 1_000_000  # 1M step a generazione (~50 min)
N_ENVS = 8
SEED_BASE = 42

# Iperparametri (gli stessi del v5-stable, sicuri)
LR = 1e-4
N_STEPS = 2048
BATCH_SIZE = 128
ENT_COEF = 0.05

# Probabilita' di campionare un avversario random invece che dalla popolation.
# 0.0 = sempre RL avversari (puro self-play)
# 0.3 = 30% delle volte avversario random (curriculum learning)
P_AVVERSARIO_RANDOM = 0.2

# Stage A2 attivo? Per ora NO, partiamo dalla baseline 318 feature.
# Cambiare a True solo quando avremo testato Stage A2 in self-play.
USE_STAGE_A2 = False

print(f'N generazioni: {N_GENERAZIONI}')
print(f'Steps per gen: {STEPS_PER_GENERAZIONE:,}')
print(f'P avversario random: {P_AVVERSARIO_RANDOM}')
print(f'Stage A2: {USE_STAGE_A2}')

## Cella 3 — Inizializza population con la baseline

In [ ]:
import shutil

# Copia baseline come gen0 nella population
gen0_path = f'{POPULATION_DIR}/gen0.zip'
if not os.path.exists(gen0_path):
    shutil.copy(BASELINE_PATH, gen0_path)
    print(f'Copiata baseline come gen0: {gen0_path}')
else:
    print(f'gen0 gia esiste: {gen0_path}')

# Lista population
import glob
population_files = sorted(glob.glob(f'{POPULATION_DIR}/gen*.zip'))
print(f'\nPopulation attuale ({len(population_files)} membri):')
for p in population_files:
    print(f'  {os.path.basename(p)}')

## Cella 4 — Funzioni helper per self-play

In [ ]:
from risiko_env import encoding as _encoding
from risiko_env import RisikoEnv
from sb3_contrib import MaskablePPO
from sb3_contrib.common.wrappers import ActionMasker
from stable_baselines3.common.vec_env import SubprocVecEnv, VecMonitor
from stable_baselines3.common.callbacks import CheckpointCallback
import random

def carica_population():
    """Carica TUTTI i membri della population dalla cartella."""
    files = sorted(glob.glob(f'{POPULATION_DIR}/gen*.zip'))
    return [MaskablePPO.load(f) for f in files], files

def mask_fn(env):
    return env._costruisci_info()['action_mask']

def make_train_env(seed_offset, population, p_random):
    """
    Crea un env per training. Per ogni env, gli avversari sono campionati
    a runtime: con probabilita p_random sono random, altrimenti dalla population.
    
    NB: la SubprocVecEnv ricarica i moduli, quindi i 'modelli' devono essere
    passati come PATH e ricaricati nel subprocess.
    """
    population_paths = sorted(glob.glob(f'{POPULATION_DIR}/gen*.zip'))
    
    def _init():
        from risiko_env import encoding as _enc
        _enc.STAGE_A_ATTIVO = USE_STAGE_A2
        
        # Per ogni env carico la population (subprocess)
        from sb3_contrib import MaskablePPO as _MaskablePPO
        avversari_modelli = [_MaskablePPO.load(p) for p in population_paths]
        
        # Costruisco gli avversari per questo env
        rnd = random.Random(seed_offset)
        avversari = {}
        for col in ['ROSSO', 'VERDE', 'GIALLO']:
            if rnd.random() < p_random:
                avversari[col] = None  # random
            else:
                avversari[col] = rnd.choice(avversari_modelli)
        
        env = RisikoEnv(seed=SEED_BASE + seed_offset, avversari=avversari)
        env = ActionMasker(env, mask_fn)
        return env
    return _init

print('Helper functions ready')

## Cella 5 — Loop di self-play

In [ ]:
import time

for gen_n in range(1, N_GENERAZIONI + 1):
    print(f'\n{"="*70}')
    print(f'  GENERAZIONE {gen_n} / {N_GENERAZIONI}')
    print(f'{"="*70}\n')
    
    # Lista population corrente (gen0..gen{n-1})
    population_paths = sorted(glob.glob(f'{POPULATION_DIR}/gen*.zip'))
    print(f'Population corrente ({len(population_paths)} membri):')
    for p in population_paths:
        print(f'  {os.path.basename(p)}')
    
    # Crea env vettorizzato
    env = SubprocVecEnv([make_train_env(i, None, P_AVVERSARIO_RANDOM) for i in range(N_ENVS)])
    env = VecMonitor(env)
    
    # Crea modello (parte da gen{n-1} se esiste, altrimenti da scratch)
    seed_modello = SEED_BASE + gen_n  # seed diverso per ogni gen
    model = MaskablePPO(
        'MlpPolicy',
        env,
        learning_rate=LR,
        n_steps=N_STEPS,
        batch_size=BATCH_SIZE,
        ent_coef=ENT_COEF,
        verbose=1,
        seed=seed_modello,
        tensorboard_log=f'{CHECKPOINT_DIR}/tensorboard',
    )
    
    # Inizializza dai pesi della gen precedente (transfer learning)
    last_gen = population_paths[-1]
    print(f'\nCarico pesi iniziali da: {os.path.basename(last_gen)}')
    try:
        prev_model = MaskablePPO.load(last_gen)
        # Trasferisci pesi se observation space coincide
        if prev_model.observation_space.shape == model.observation_space.shape:
            model.set_parameters(prev_model.get_parameters())
            print('Transfer learning OK')
        else:
            print(f'Observation space diverso: prev={prev_model.observation_space.shape} vs new={model.observation_space.shape}. Training da scratch.')
    except Exception as e:
        print(f'Transfer learning fallito: {e}. Training da scratch.')
    
    # Checkpoint periodico
    cb = CheckpointCallback(
        save_freq=200_000 // N_ENVS,
        save_path=f'{POPULATION_DIR}/gen{gen_n}_checkpoints/',
        name_prefix=f'gen{gen_n}',
    )
    
    # Train
    t0 = time.time()
    model.learn(
        total_timesteps=STEPS_PER_GENERAZIONE,
        callback=cb,
        progress_bar=True,
    )
    durata = (time.time() - t0) / 60
    
    # Salva nuovo membro popolation
    new_gen_path = f'{POPULATION_DIR}/gen{gen_n}.zip'
    model.save(new_gen_path)
    print(f'\n✓ Salvato gen{gen_n}: {new_gen_path}')
    print(f'  Durata: {durata:.1f} min')
    
    # Cleanup env
    env.close()
    del env, model
    
print(f'\n{"="*70}')
print(f'  TRAINING COMPLETATO: {N_GENERAZIONI} generazioni')
print(f'{"="*70}')

## Cella 6 — Validazione: ogni gen vs ogni gen (round-robin)

In [ ]:
import numpy as np
from risiko_env import encoding as _encoding
_encoding.STAGE_A_ATTIVO = USE_STAGE_A2

population_paths = sorted(glob.glob(f'{POPULATION_DIR}/gen*.zip'))
population = [(os.path.basename(p)[:-4], MaskablePPO.load(p)) for p in population_paths]

N_PARTITE_VALID = 50  # rapide per matrice

print(f'Round-robin: {len(population)} bot × {len(population)-1} avversari × {N_PARTITE_VALID} partite\n')

matrice = {}
for nome_a, model_a in population:
    for nome_b, model_b in population:
        if nome_a == nome_b:
            continue
        # A è il bot principale (BLU). B controlla ROSSO+VERDE+GIALLO.
        wins_a = 0
        for s in range(N_PARTITE_VALID):
            env = RisikoEnv(seed=s, avversari={'ROSSO': model_b, 'VERDE': model_b, 'GIALLO': model_b})
            obs, info = env.reset()
            n_step = 0
            while n_step < 5000:
                mask = info['action_mask']
                action, _ = model_a.predict(obs, action_masks=mask, deterministic=True)
                obs, reward, term, trunc, info = env.step(int(action))
                n_step += 1
                if term or trunc: break
            if reward == 1.0:
                wins_a += 1
        matrice[(nome_a, nome_b)] = wins_a / N_PARTITE_VALID
        print(f'  {nome_a} vs 3×{nome_b}: {wins_a}/{N_PARTITE_VALID} ({wins_a/N_PARTITE_VALID*100:.1f}%)')

# Stampa matrice
print(f'\n{"="*70}')
print('MATRICE WIN RATE (riga = bot principale, colonna = avversari)')
print(f'{"="*70}\n')
names = [n for n, _ in population]
header = '         | ' + ' | '.join(f'{n:>6}' for n in names)
print(header)
print('-' * len(header))
for nome_a, _ in population:
    riga = f'{nome_a:>8} | '
    for nome_b, _ in population:
        if nome_a == nome_b:
            cella = '  --  '
        else:
            wr = matrice.get((nome_a, nome_b), 0)
            cella = f'{wr*100:>5.1f}%'
        riga += f'{cella:>6} | '
    print(riga)

## Cella 7 — Sanity check: ogni gen vs random

In [ ]:
N_PARTITE_VS_RANDOM = 100

print(f'Each gen vs 3×random ({N_PARTITE_VS_RANDOM} partite):\n')
for nome, model in population:
    wins = 0
    for s in range(N_PARTITE_VS_RANDOM):
        env = RisikoEnv(seed=s)  # default = random
        obs, info = env.reset()
        n_step = 0
        while n_step < 5000:
            mask = info['action_mask']
            action, _ = model.predict(obs, action_masks=mask, deterministic=True)
            obs, reward, term, trunc, info = env.step(int(action))
            n_step += 1
            if term or trunc: break
        if reward == 1.0:
            wins += 1
    print(f'  {nome}: {wins}/{N_PARTITE_VS_RANDOM} = {wins/N_PARTITE_VS_RANDOM*100:.1f}%')